In [5]:
import pyautogui as pyg
import time

def main():
    print("【Chrome 小恐龍自動化腳本】啟動！")
    print("請在 5 秒內，將滑鼠游標移到「小恐龍嘴部的深色像素」上，不要移動...")
    time.sleep(5)
    
    dino_x, dino_y = pyg.position()
    print(f"✅ 基準點已鎖定: X={dino_x}, Y={dino_y}")
    print("遊戲即將開始！若要手動中斷，請將滑鼠狂掃至螢幕最「左上角」。")
    
    pyg.press('space')
    time.sleep(1)
    
    start_time = time.time()
    last_obs_x = -1
    static_count = 0

    try:
        while True:
            # 【優化 3】強制手動檢查防呆機制：只要滑鼠碰到左上角 (X<=5, Y<=5) 絕對秒停
            mouse_x, mouse_y = pyg.position()
            if mouse_x <= 5 and mouse_y <= 5:
                print("\n🛑 已手動觸發左上角緊急停止，程式安全結束！")
                break

            elapsed_time = time.time() - start_time
            
            # 【優化 1】針對大後期：加大預判增長係數(3.5)，並將預判極限拉高至 350 像素
            lookahead_dist = int(90 + (elapsed_time * 3.5))
            if lookahead_dist > 350: 
                lookahead_dist = 350
                
            # 維持原本的偵測區域設定
            box_x = dino_x + 40
            box_y = dino_y - 5 
            box_w = lookahead_dist
            box_h = 30
            
            im = pyg.screenshot(region=(box_x, box_y, box_w, box_h))
            
            # 取第一點作為背景參考色
            bg_color = im.getpixel((0, 0))
            bg_r = bg_color[0]
            
            jump = False
            current_obs_x = -1
            
            # 步進掃描，追求極致效能
            for x in range(0, box_w, 4):
                for y in range(0, box_h, 4):
                    r = im.getpixel((x, y))[0]
                    
                    # 【優化 2】針對日夜交替漸層期：將色差門檻降至 40
                    # 這樣即使背景變成過渡期的灰色，依然能抓出深色/淺色的障礙物，且不會抓到雲朵
                    if abs(r - bg_r) > 40:
                        jump = True
                        current_obs_x = x
                        break
                if jump:
                    break
            
            # 遊戲結束判定 (連續 15 次畫面凍結)
            if current_obs_x != -1 and current_obs_x == last_obs_x:
                static_count += 1
            else:
                static_count = 0
                last_obs_x = current_obs_x
                
            if static_count > 15:
                print(f"💀 偵測到撞擊！遊戲結束。共存活了 {elapsed_time:.1f} 秒。")
                break
            
            # 執行跳躍
            if jump:
                pyg.press('space')
                # 【優化 1-補充】針對後期密集障礙物：將最小冷卻時間下修至 0.08 秒
                # 確保在極高車速下，恐龍一落地就能馬上再跳
                cooldown = max(0.08, 0.40 - (elapsed_time * 0.0025))
                time.sleep(cooldown)

    except Exception as e:
        print(f"\n⚠️ 發生未預期的錯誤結束: {e}")

if __name__ == '__main__':
    main()

【Chrome 小恐龍自動化腳本】啟動！
請在 5 秒內，將滑鼠游標移到「小恐龍嘴部的深色像素」上，不要移動...
✅ 基準點已鎖定: X=1361, Y=575
遊戲即將開始！若要手動中斷，請將滑鼠狂掃至螢幕最「左上角」。

🛑 已手動觸發左上角緊急停止，程式安全結束！
